In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType, IntegerType
from pyspark.sql.functions import col, regexp_replace, nullif, lit, to_date
from pyspark.sql.types import DecimalType

def clean_decimal(c, p=38, s=8):
    cleaned = regexp_replace(col(c), "[^0-9.]", "")
    return nullif(cleaned, lit("")).alias(c)
def clean_int(c):
    cleaned = regexp_replace(col(c), "[^0-9.]", "")
    return when(
        length(cleaned) == 0,
        None
    ).otherwise(
        cleaned.cast(IntegerType())
    ).alias(c)
def clean_short(c):
    cleaned = regexp_replace(col(c), "[^0-9.]", "")
    return when(
        length(cleaned) == 0,
        None
    ).otherwise(
        cleaned.cast("short")
    ).alias(c)
def clean_date(c):
    return coalesce(
        expr(f"try_to_date({c}, 'M/d/yyyy')"),
        expr(f"try_to_date({c}, 'MM/dd/yyyy')"),
        expr(f"try_to_date({c}, 'yyyy-MM-dd')"),
        expr(f"try_to_date({c}, 'yyyyMMdd')")
    ).alias(c)
df = (
    spark.read
    .option("header", "true")
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .csv("/Volumes/mb_poc/data_raw/uc2/Book3 (1).csv")
)

df_tgt = (
    df.select(
        col("SUB_RGON").alias("SUB_RGON"),
        clean_date("CDR_DT"),
        col("MA_CNC1").alias("MA_CNC1"),  
        col("TEN_CNC1").alias("TEN_CNC1"),
        col("MA_CNC2").alias("MA_CNC1"),
        col("TEN_CNC2").alias("TEN_CNC2"),
        col("RM_CODE").alias("RM_CODE"),
        col("RM_NM").alias("RM_NM"),
        col("TTL_NM").alias("TTL_NM"),
        clean_int("NBR_OF_NEW_AC_MTD"),
        clean_decimal("NEW_OPN_TVR_MTD"),
        clean_decimal("SP_SO_LS_THUONG"),
        clean_decimal("SP_SO_LS_VOUCHER"),
        clean_decimal("SP_QUAY_LS_THUONG"),
        clean_decimal("SP_QUAY_LS_VOUCHER")
    )
)
#df_tgt.printSchema()
#df_tgt.display()
#df_tgt.explain(True)

df_tgt.write.mode("append").insertInto("mb_poc.gold.rpt_khcn_hdv_d09_dgtal_svg_details")

